# Day 5 – quiz pictures from the 5 website scans
1. **Runtime → Run all** is clicked.
2. In CELL 1, **Choose Files** → `web_data_final.zip` (from Day 2).
3. CELL 3 must show `5 usable scans`; CELL 4 shows the numbering.
4. CELL 7 shows the check sheet and downloads `quiz_data.zip` for Intern B.
5. Only CELL 2 is changed after the check (LABEL_MODE, SKIP, USE_CASE).

Every program line has its explanation on its right, after `#`.

In [ ]:
# ---- CELL 1: the website data zip is uploaded
try:                                                          # the Colab upload tool is tried first
    from google.colab import files                            # Colab's tool that moves files from the laptop into Colab
    uploaded = files.upload()                                 # "Choose Files" opens: web_data_final.zip is picked (from Day 2)
    print("Files received:", list(uploaded))                  # the names of the arrived files are shown
except ImportError:                                           # this happens only outside Colab (for example on a laptop)
    print("Not in Colab: web_data_final.zip is read from this folder")  # the zip must be next to this program

In [ ]:
# ---- CELL 2: settings (only this cell is typed in)
LABEL_MODE = "auto"                                           # "auto" = found by the program; "vista" = numbers 1, 3, 4 ... 15; "compact" = numbers 1–12
VIEWS = ["axial", "coronal"]                                  # two pictures per organ: axial (seen from the feet) and coronal (seen from the front)
WINDOW = (-160, 240)                                          # CT window in HU for soft tissue (level 40, width 400)
OUTLINE = "yellow"                                            # colour of the organ outline
SKIP = []                                                     # pictures left out after the check, e.g. ["pancreas_coronal"]
USE_CASE = {}                                                 # another scan for one organ after the check, e.g. {"pancreas": "s0024"}

In [ ]:
# ---- CELL 3: the zip is unpacked and the files of every scan are found
import glob, json, os, re, zipfile                            # file search, JSON, file times, name patterns, zip files
import numpy as np                                            # numbers and arrays
import nibabel as nib                                         # reads NIfTI scan files (.nii and .nii.gz)
from nibabel.processing import resample_from_to               # moves a mask onto the CT grid if the grids differ

zips = sorted(glob.glob("web_data_final*.zip"), key=os.path.getmtime)  # the uploaded zip(s), newest last
if not zips:                                                  # nothing was uploaded
    raise FileNotFoundError("web_data_final.zip is missing")  # the program stops with a clear message
with zipfile.ZipFile(zips[-1]) as z:                          # the newest zip is opened
    z.extractall("web_data")                                  # and unpacked into the folder web_data
ROLE_WORDS = {"ai": ("vista", "_ai", "ai_", "pred"), "expert": ("expert", "reference", "gt", "label", "truth", "seg"), "ct": ("ct", "image", "img")}  # words in file names

def role_of(path):                                            # decides: CT image, expert mask or AI mask
    name = os.path.basename(path).lower()                     # only the file name, in small letters
    for role, words in ROLE_WORDS.items():                    # AI words are tested first, then expert, then CT
        if any(w in name for w in words):                     # one of the words is in the name
            return role                                       # the role is found
    return "?"                                                # an unknown file

def case_of(path):                                            # finds the scan ID (for example s0011) in the path
    m = re.search(r"s\d{4}", path)                            # TotalSegmentator IDs look like s0011
    return m.group(0) if m else os.path.basename(os.path.dirname(path))  # otherwise the folder name is used

cases = {}                                                    # scan ID → {"ct": file, "expert": file, "ai": file}
for f in sorted(glob.glob("web_data/**/*.nii*", recursive=True)):  # every NIfTI file in the zip, in any sub-folder
    cases.setdefault(case_of(f), {}).setdefault(role_of(f), f)  # kept under its scan and role (the first one wins)
for cid, roles in cases.items():                              # every scan
    if "ct" not in roles and "?" in roles:                    # no CT word was found, but there is one unknown file
        roles["ct"] = roles.pop("?")                          # the unknown file is taken as the CT
    print(cid, {r: os.path.basename(p) for r, p in roles.items()})  # what was found is shown
cases = {c: r for c, r in cases.items() if "ct" in r and "expert" in r}  # only scans with a CT and an expert mask are used
if not cases:                                                 # nothing usable was found
    raise ValueError("No scan has both a CT and an expert mask: the list above is sent to DrRB")  # a clear stop
print(len(cases), "usable scans")                             # usually 5

In [ ]:
# ---- CELL 4: the label numbers are matched to organ names
VISTA = {1: "Liver", 3: "Spleen", 4: "Pancreas", 5: "Right kidney", 14: "Left kidney", 10: "Gallbladder", 12: "Stomach", 6: "Aorta", 7: "Inferior vena cava", 11: "Oesophagus", 13: "Duodenum", 15: "Urinary bladder"}  # VISTA-3D numbers of our 12 organs
COMPACT = {i + 1: name for i, name in enumerate(VISTA.values())}  # the same 12 organs numbered 1–12 in the same order
first = nib.load(next(iter(cases.values()))["expert"])        # the expert mask of the first scan
values = {int(v) for v in np.unique(np.asarray(first.dataobj))} - {0}  # the label numbers used (0 = background)
mode = LABEL_MODE if LABEL_MODE != "auto" else ("vista" if max(values) > 12 else "compact")  # numbers above 12 exist only in VISTA numbering
NAMES = VISTA if mode == "vista" else COMPACT                 # the chosen table
print("Label numbers found:", sorted(values), "→ numbering:", mode)  # shown, so that it can be checked
print("Not used:", sorted(values - set(NAMES)) or "none")     # numbers that are not one of our 12 organs are ignored

In [ ]:
# ---- CELL 5: for every organ, the scan and the slice are chosen
def load_case(cid):                                           # loads the CT and the expert mask of one scan on the same grid
    ct = nib.as_closest_canonical(nib.load(cases[cid]["ct"]))  # the CT, turned to the standard direction (RAS)
    mk = nib.as_closest_canonical(nib.load(cases[cid]["expert"]))  # the expert mask, turned the same way
    if mk.shape != ct.shape or not np.allclose(mk.affine, ct.affine, atol=1e-3):  # the mask grid differs from the CT grid
        mk = resample_from_to(mk, ct, order=0)                # the mask is moved onto the CT grid (order 0 keeps whole numbers)
    return ct, np.asarray(ct.dataobj, dtype=np.float32), np.rint(np.asarray(mk.dataobj)).astype(np.int16)  # image, CT numbers, mask numbers

data = {cid: load_case(cid) for cid in cases}                 # all scans are loaded once (5 small 2 mm scans fit easily)
plan = []                                                     # the list of pictures to make
for num, organ in NAMES.items():                              # every organ
    sizes = {cid: int((d[2] == num).sum()) for cid, d in data.items()}  # how many voxels of this organ each scan has
    cid = USE_CASE.get(organ.lower(), max(sizes, key=sizes.get))  # the scan with the largest organ (or the one set in CELL 2)
    if cid not in data or sizes[cid] == 0:                    # a wrong ID in CELL 2, or the organ is in no scan
        print("Skipped (not found):", organ, cid)             # shown, and this organ is left out
        continue                                              # go to the next organ
    organ_mask = data[cid][2] == num                          # True where this organ is, in the chosen scan
    for view in VIEWS:                                        # axial and coronal
        axis = 2 if view == "axial" else 1                    # axial slices run along z (axis 2), coronal slices along y (axis 1)
        area = organ_mask.sum(axis=tuple(a for a in range(3) if a != axis))  # the organ area in every slice
        plan.append((organ, num, cid, view, int(np.argmax(area))))  # the slice with the biggest organ area is used
print(len(plan), "pictures planned")                          # usually 24

In [ ]:
# ---- CELL 6: the pictures are drawn and saved
import matplotlib                                             # the drawing library
matplotlib.use("Agg")                                         # pictures are drawn into files, not onto a screen
import matplotlib.pyplot as plt                               # the drawing commands

def show2d(volume, axis, index):                              # one slice, turned like a normal radiology picture
    s = np.take(volume, index, axis=axis)                     # the slice is cut out of the 3-D volume
    return np.fliplr(np.rot90(s))                             # front (or head) at the top; patient's right on the left side

os.makedirs("quiz/img", exist_ok=True)                        # the output folder quiz/img is made
questions = []                                                # the entries for quiz_questions.json
for organ, num, cid, view, idx in plan:                       # one picture per plan entry
    slug = organ.lower().replace(" ", "_") + "_" + view       # file name, e.g. right_kidney_axial
    if slug in SKIP:                                          # left out after the check (CELL 2)
        continue                                              # go to the next picture
    img, ct, mk = data[cid]                                   # the scan data
    axis = 2 if view == "axial" else 1                        # the same axis as in CELL 5
    zx, zy, zz = img.header.get_zooms()[:3]                   # voxel size in mm along x, y and z
    aspect = (zy if view == "axial" else zz) / zx             # keeps the picture from looking squashed
    fig, ax = plt.subplots(figsize=(5, 5), dpi=110)           # a square picture of about 550 × 550 pixels
    ax.imshow(show2d(ct, axis, idx), cmap="gray", vmin=WINDOW[0], vmax=WINDOW[1], aspect=aspect)  # the CT slice in grey
    ax.contour(show2d(mk == num, axis, idx).astype(float), levels=[0.5], colors=OUTLINE, linewidths=1.6)  # the organ outline
    ax.axis("off")                                            # no axes or numbers around the picture
    fig.savefig(f"quiz/img/{slug}.png", bbox_inches="tight", pad_inches=0, facecolor="black")  # saved as a PNG file
    plt.close(fig)                                            # the picture is closed to free memory
    questions.append({"image": f"img/{slug}.png", "view": view, "answer": organ, "case": cid, "slice": idx})  # one quiz question
print(len(questions), "pictures saved in quiz/img")           # usually 24

In [ ]:
# ---- CELL 7: the quiz file, the check sheet and the zip are made
quiz = {"version": "draft", "made": "", "note": "Draft quiz – to be checked by faculty", "organs": list(NAMES.values()), "facts": {}, "questions": questions}  # the quiz file content
with open("quiz/quiz_questions.json", "w", encoding="utf-8") as f:  # the quiz file is opened for writing
    json.dump(quiz, f, indent=1, ensure_ascii=False)          # and written (the dash – stays readable)
cols = 6                                                      # the check sheet has 6 pictures per row
rows = -(-len(questions) // cols)                             # the rows needed, rounded up
fig, axes = plt.subplots(rows, cols, figsize=(cols * 2.2, rows * 2.4))  # one small panel per picture
for ax in np.ravel(axes):                                     # every panel
    ax.axis("off")                                            # no axes
for ax, q in zip(np.ravel(axes), questions):                  # one panel per question
    ax.imshow(plt.imread("quiz/" + q["image"]))               # the picture
    ax.set_title(f"{q['answer']}\n{q['view']} · {q['case']}", fontsize=7)  # answer, view and scan as the title
fig.savefig("quiz_check_sheet.png", dpi=110, bbox_inches="tight")  # the check sheet is saved
plt.close(fig)                                                # closed to free memory
with zipfile.ZipFile("quiz_data.zip", "w") as z:              # one zip for Intern B
    for f in sorted(glob.glob("quiz/**/*", recursive=True)):  # every file in the quiz folder
        if os.path.isfile(f):                                 # folders themselves are not added
            z.write(f)                                        # the file goes into the zip with its path (quiz/...)
    z.write("quiz_check_sheet.png")                           # the check sheet is added for Intern A
try:                                                          # the next lines work only in Colab
    from IPython.display import Image, display                # tools that show a picture in the notebook
    display(Image("quiz_check_sheet.png"))                    # the check sheet is shown below this cell
    files.download("quiz_data.zip")                           # the zip goes to the laptop's Downloads folder
except (ImportError, NameError):                              # outside Colab these tools do not exist
    print("Saved in this folder: quiz_data.zip and quiz_check_sheet.png")  # a message for laptop use